In [0]:
import numpy as np
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient


# ============================================================
# 1. DATA PREPARER (loads tables, separates X and y)
# ============================================================

class ModelDataPreparer:

    def __init__(self, spark, train_table, validation_table, test_table,
                 label_column, exclude_columns):
        self.spark = spark
        self.train_table = train_table
        self.validation_table = validation_table
        self.test_table = test_table
        self.label_column = label_column
        self.exclude_columns = exclude_columns
        self.feature_columns = None

    def get_feature_columns(self, df):
        # X = every column except the label and the excluded columns
        self.feature_columns = [
            c for c in df.columns
            if c != self.label_column and c not in self.exclude_columns
        ]

    def to_pandas_xy(self, spark_df):
        pandas_df = spark_df.select(self.feature_columns + [self.label_column]).toPandas()

        # Decimal -> float (avoids the JSON serialization error)
        for c in pandas_df.columns:
            pandas_df[c] = pandas_df[c].astype(float)

        X = pandas_df[self.feature_columns]
        y = pandas_df[self.label_column]
        return X, y

    def prepare(self):
        train_df = self.spark.table(self.train_table)
        validation_df = self.spark.table(self.validation_table)
        test_df = self.spark.table(self.test_table)

        self.get_feature_columns(train_df)
        print("Feature columns (X): " + str(self.feature_columns))
        print("Label column (Y): " + self.label_column)

        X_train, y_train = self.to_pandas_xy(train_df)
        X_validation, y_validation = self.to_pandas_xy(validation_df)
        X_test, y_test = self.to_pandas_xy(test_df)

        return X_train, y_train, X_validation, y_validation, X_test, y_test


# ============================================================
# 2. TRAINER (trains and evaluates the model)
# ============================================================

class RandomForestTrainer:

    def __init__(self, num_trees=100, max_depth=8):
        self.num_trees = num_trees
        self.max_depth = max_depth
        self.model = None

    def train(self, X_train, y_train):
        self.model = RandomForestRegressor(
            n_estimators=self.num_trees,
            max_depth=self.max_depth,
            random_state=42
        )
        self.model.fit(X_train, y_train)
        return self.model

    def evaluate(self, X, y, dataset_name):
        predictions = self.model.predict(X)

        rmse = np.sqrt(mean_squared_error(y, predictions))
        mae = mean_absolute_error(y, predictions)
        r2 = r2_score(y, predictions)

        print(dataset_name + " -> RMSE: " + str(rmse) + " | MAE: " + str(mae) + " | R2: " + str(r2))
        return {"rmse": rmse, "mae": mae, "r2": r2}


# ============================================================
# 3. REGISTRAR (registers the model only if thresholds are met)
# ============================================================

class ModelRegistrar:

    def __init__(self, model_name, max_rmse, min_r2):
        self.model_name = model_name
        self.max_rmse = max_rmse
        self.min_r2 = min_r2

    def register_if_good(self, model_uri, rmse, r2):
        # Returns the new version number, or None if thresholds were not met
        rmse_ok = rmse < self.max_rmse
        r2_ok = r2 > self.min_r2

        if rmse_ok and r2_ok:
            registered = mlflow.register_model(model_uri=model_uri, name=self.model_name)
            print("Registered as version " + str(registered.version))
            return registered.version

        print("Not registered. Thresholds not met:")
        print("  RMSE: " + str(rmse) + " (needs < " + str(self.max_rmse) + ") -> " + ("OK" if rmse_ok else "FAIL"))
        print("  R2: " + str(r2) + " (needs > " + str(self.min_r2) + ") -> " + ("OK" if r2_ok else "FAIL"))
        return None


# ============================================================
# 4. CHAMPION / CHALLENGER MANAGER
# ============================================================

class ChampionChallengerManager:

    def __init__(self, model_name, champion_alias="champion", challenger_alias="challenger"):
        self.model_name = model_name
        self.champion_alias = champion_alias
        self.challenger_alias = challenger_alias
        self.client = MlflowClient()

    def _get_rmse_by_alias(self, alias):
        # Returns (version, validation_rmse) for whichever version holds this alias
        # Returns (None, None) if the alias does not exist yet
        try:
            version = self.client.get_model_version_by_alias(self.model_name, alias)
        except Exception:
            return None, None

        run = self.client.get_run(version.run_id)
        rmse = run.data.metrics.get("validation_rmse")
        return version.version, rmse

    def promote_challenger_if_better(self, new_version, new_rmse):
        # Step 1: the newly registered model always becomes the challenger
        self.client.set_registered_model_alias(self.model_name, self.challenger_alias, new_version)
        print("Version " + str(new_version) + " set as '" + self.challenger_alias + "'")

        # Step 2: does a champion already exist?
        champion_version, champion_rmse = self._get_rmse_by_alias(self.champion_alias)

        if champion_version is None:
            self.client.set_registered_model_alias(self.model_name, self.champion_alias, new_version)
            print("No existing champion. Version " + str(new_version) + " promoted to '" + self.champion_alias + "'")
            return

        # Step 3: compare the challenger's RMSE against the champion's RMSE
        print("Champion (v" + str(champion_version) + ") validation RMSE: " + str(champion_rmse))
        print("Challenger (v" + str(new_version) + ") validation RMSE: " + str(new_rmse))

        if new_rmse < champion_rmse:
            self.client.set_registered_model_alias(self.model_name, self.champion_alias, new_version)
            print("Version " + str(new_version) + " promoted to '" + self.champion_alias + "'")
        else:
            print("Version " + str(new_version) + " remains '" + self.challenger_alias +
                  "', current champion (v" + str(champion_version) + ") is not beaten")


# ============================================================
# 5. CONFIGURATION
# ============================================================

label_column = "total_demand_quantity"

exclude_columns = [
    "transaction_date",
    "product_name",
    "destination_city",
    "product_name_index",
    "destination_city_index"
]

model_name = "oil_gas_mlops.ml_models.supply_chain_demand_model_sklearn"
num_trees = 110
max_depth = 8

max_rmse = 1000
min_r2 = 0.85


# ============================================================
# 6. PREPARE DATA
# ============================================================

preparer = ModelDataPreparer(
    spark=spark,
    train_table="oil_gas_mlops.ml_data.train_data",
    validation_table="oil_gas_mlops.ml_data.validation_data",
    test_table="oil_gas_mlops.ml_data.test_data",
    label_column=label_column,
    exclude_columns=exclude_columns
)

X_train, y_train, X_validation, y_validation, X_test, y_test = preparer.prepare()


# ============================================================
# 7. MLFLOW SETUP
# ============================================================

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/hsshinnde0702@gmail.com/rf_supply_chain")

run_name = "rf_supply_chain_" + datetime.now().strftime("%Y%m%d_%H%M%S")


# ============================================================
# 8. TRAIN, EVALUATE, LOG, REGISTER, CHAMPION/CHALLENGER
# ============================================================

with mlflow.start_run(run_name=run_name) as run:

    # ---- Parameters ----
    mlflow.log_param("model_type", "RandomForestRegressor (sklearn)")
    mlflow.log_param("num_trees", num_trees)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("label_column", label_column)
    mlflow.log_param("num_features", len(preparer.feature_columns))

    # ---- Tags ----
    mlflow.set_tag("project", "oil_gas_supply_chain")
    mlflow.set_tag("problem_type", "time_series_regression")
    mlflow.set_tag("stage", "development")

    # ---- Train ----
    trainer = RandomForestTrainer(num_trees=num_trees, max_depth=max_depth)
    model = trainer.train(X_train, y_train)

    # ---- Evaluate ----
    train_metrics = trainer.evaluate(X_train, y_train, "Train")
    validation_metrics = trainer.evaluate(X_validation, y_validation, "Validation")
    test_metrics = trainer.evaluate(X_test, y_test, "Test")

    # ---- Log metrics ----
    mlflow.log_metric("train_rmse", train_metrics["rmse"])
    mlflow.log_metric("train_mae", train_metrics["mae"])
    mlflow.log_metric("train_r2", train_metrics["r2"])

    mlflow.log_metric("validation_rmse", validation_metrics["rmse"])
    mlflow.log_metric("validation_mae", validation_metrics["mae"])
    mlflow.log_metric("validation_r2", validation_metrics["r2"])

    mlflow.log_metric("test_rmse", test_metrics["rmse"])
    mlflow.log_metric("test_mae", test_metrics["mae"])
    mlflow.log_metric("test_r2", test_metrics["r2"])

    # ---- Signature ----
    signature_input = X_train.head(10)
    signature_output = model.predict(signature_input)
    signature = infer_signature(signature_input, signature_output)

    # ---- Log model (saved with this run, NOT registered yet) ----
    model_info = mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=signature_input
    )

    # ---- Step 1: register only if thresholds are met ----
    registrar = ModelRegistrar(model_name=model_name, max_rmse=max_rmse, min_r2=min_r2)
    new_version = registrar.register_if_good(
        model_uri=model_info.model_uri,
        rmse=validation_metrics["rmse"],
        r2=validation_metrics["r2"]
    )

    # ---- Step 2: if registered, run champion/challenger logic ----
    if new_version is not None:
        cc_manager = ChampionChallengerManager(model_name=model_name)
        cc_manager.promote_challenger_if_better(
            new_version=new_version,
            new_rmse=validation_metrics["rmse"]
        )

    print("Run ID: " + run.info.run_id)